In [73]:
import re
import pathlib
import json
import numpy as np
import pandas as pd

In [74]:
def filereader_funktion(x):
    
    with open(x, "rb") as file:
        file_text_content = file.read()
        
        if file_text_content[:3] == b"\xef\xbb\xbf":
            encoding_used = "UTF-8"
        else:
            encoding_used = "ISO-8859-1/Latin-1"

        return file_text_content, encoding_used

In [75]:
filereader_funktion(pathlib.Path("data/2026/library_cafe/POS_01/register_sales.log"))

(b'[DEBUG] BARCODE SCANNER REBOOTED\r\n[2026-10-15 07:00:36] [POS_01] item=Cappuccino price=4.38 NOK (SUCCESS)\r\n[2026-10-15 07:01:02] [POS_01] item=Croissant price=3.89 EUR (SUCCESS)\r\n--- PRINTER PAPER LOW ---\r\n[2026-10-15 07:01:56] [POS_01] item=Latte price=4.97 USD (REFUND)\r\n[2026-10-15 07:02:38] [POS_01] item=Espresso price=2.67 EUR (CANCELLED)\r\n[2026-10-15 07:04:32] [POS_01] item=Muffin price=3.45 NOK (REFUND)\r\n--- PRINTER PAPER LOW ---\r\n[2026-10-15 07:05:59] [POS_01] item=Cappuccino price=4.51 EUR (SUCCESS)\r\n[2026-10-15 07:07:58] [POS_01] item=Sandwich price=7.79 NOK (SUCCESS)\r\n[2026-10-15 07:10:08] [POS_01] item=Espresso price=3.12 EUR (SUCCESS)\r\n=== NETWORK SYNC OK ===\r\n[2026-10-15 07:12:09] [POS_01] item=Cappuccino price=4.43 EUR (SUCCESS)\r\n[2026-10-15 07:13:09] [POS_01] item=Muffin price=3.96 EUR (REFUND)\r\n=== NETWORK SYNC OK ===\r\n[2026-10-15 07:14:54] [POS_01] item=Croissant price=3.89 NOK (SUCCESS)\r\n[2026-10-15 07:17:24] [POS_01] item=Sandwich p

In [76]:
def log_parser_funksjon(x):
    list_of_dictionaries = []
    dictionary = {}

    with open(x, "r") as file:
        file_text = file.read()

    for i in file_text.splitlines():
        dictionary["POS"] = re.findall(r"POS_\d{2}", i)
        match = re.search(
            r"(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})", i
        )
        dictionary["timestamp"] = match.group(1) if match else None

        match = re.search(r"item=([A-Za-z-]+)", i)
        dictionary["item"] = match.group(1) if match else None

        dictionary["currency"] = re.findall(r"EUR|NOK|USD", i)
        dictionary["status"] = re.findall(r"SUCCESS|REFUND|CANCELLED", i)

        match = re.search(r"price=([0-9.]+)", i)
        dictionary["price"] = match.group(1) if match else None

        if dictionary["timestamp"] and dictionary["item"] and dictionary["price"] and dictionary["status"]:
            list_of_dictionaries.append(dictionary.copy())

    return list_of_dictionaries

     

In [77]:
log_parser_funksjon(pathlib.Path("data/2026/library_cafe/POS_01/register_sales.log"))


[{'POS': ['POS_01'],
  'timestamp': '2026-10-15 07:00:36',
  'item': 'Cappuccino',
  'currency': ['NOK'],
  'status': ['SUCCESS'],
  'price': '4.38'},
 {'POS': ['POS_01'],
  'timestamp': '2026-10-15 07:01:02',
  'item': 'Croissant',
  'currency': ['EUR'],
  'status': ['SUCCESS'],
  'price': '3.89'},
 {'POS': ['POS_01'],
  'timestamp': '2026-10-15 07:01:56',
  'item': 'Latte',
  'currency': ['USD'],
  'status': ['REFUND'],
  'price': '4.97'},
 {'POS': ['POS_01'],
  'timestamp': '2026-10-15 07:02:38',
  'item': 'Espresso',
  'currency': ['EUR'],
  'status': ['CANCELLED'],
  'price': '2.67'},
 {'POS': ['POS_01'],
  'timestamp': '2026-10-15 07:04:32',
  'item': 'Muffin',
  'currency': ['NOK'],
  'status': ['REFUND'],
  'price': '3.45'},
 {'POS': ['POS_01'],
  'timestamp': '2026-10-15 07:05:59',
  'item': 'Cappuccino',
  'currency': ['EUR'],
  'status': ['SUCCESS'],
  'price': '4.51'},
 {'POS': ['POS_01'],
  'timestamp': '2026-10-15 07:07:58',
  'item': 'Sandwich',
  'currency': ['NOK'],
  

In [78]:
def fin_log_fil(x):
    log_fil = []
    for item in x.iterdir():
        if item.is_dir():
            for i in fin_log_fil(item):
                log_fil.append(i)
        elif item.suffix == ".log":
            log_fil.append(item)

    return log_fil

In [79]:
fin_log_fil(pathlib.Path("data"))

[WindowsPath('data/2026/library_cafe/POS_01/register_sales.log'),
 WindowsPath('data/2026/library_cafe/POS_02/register_sales.log'),
 WindowsPath('data/2026/library_cafe/POS_03/register_sales.log'),
 WindowsPath('data/2026/science_building/POS_01/register_sales.log'),
 WindowsPath('data/2026/science_building/POS_02/register_sales.log'),
 WindowsPath('data/2026/science_building/POS_03/register_sales.log'),
 WindowsPath('data/2026/student_union/POS_01/register_sales.log'),
 WindowsPath('data/2026/student_union/POS_02/register_sales.log'),
 WindowsPath('data/2026/student_union/POS_03/register_sales.log')]

In [80]:
def diconery_unpacker(x):
    ider = []

    if isinstance(x, dict):
        if x.get("active") is True and "id" in x:
            ider.append(x["id"])

        for value in x.values():
            ider.extend(diconery_unpacker(value))

    elif isinstance(x, list):
        for item in x:
            ider.extend(diconery_unpacker(item))

    return ider


def Meta_data(x):
    with open(x) as fil:
        data=json.load(fil)
    return diconery_unpacker(data)


In [81]:
print(Meta_data(pathlib.Path("data/store_config.json")))

['POS_01', 'POS_02', 'POS_03']


In [82]:
def hent_loggdata_uten_avvik(x):
    alle_data = []

    for fil in fin_log_fil(x):
        data = log_parser_funksjon(fil)
        alle_data.extend(data)
    
    priser = []
    for i in alle_data:
        priser.append(float(i["price"][0]))

    priser = np.array(priser)

    mean = np.mean(priser[(priser != 999)])
    std = np.std(priser[(priser != 999)])

    for i in range(len(priser)):
        if abs(priser[i] - mean) > 3 * std:
            
            priser[i]=mean

    return priser

In [83]:
hent_loggdata_uten_avvik(pathlib.Path("data"))

array([4.        , 3.        , 4.        , 2.        , 3.        ,
       4.        , 7.        , 3.        , 4.        , 3.        ,
       3.        , 7.        , 4.        , 4.        , 3.        ,
       3.99027778, 3.        , 3.        , 3.        , 4.        ,
       4.        , 4.        , 4.        , 3.99027778, 3.        ,
       2.        , 3.        , 3.        , 2.        , 4.        ,
       3.        , 4.        , 4.        , 3.        , 3.        ,
       7.        , 7.        , 4.        , 7.        , 3.        ,
       4.        , 4.        , 7.        , 2.        , 7.        ,
       7.        , 3.        , 3.        , 3.        , 7.        ,
       3.        , 3.        , 2.        , 2.        , 3.        ,
       3.        , 4.        , 3.        , 7.        , 3.        ,
       7.        , 3.        , 3.        , 3.        , 3.        ,
       3.        , 4.        , 4.        , 4.        , 2.        ,
       2.        , 3.        , 7.        , 7.        , 3.     

In [84]:
#3.2
def compute_refund_ratio(
    sales_count: np.ndarray, 
    refund_count: np.ndarray) -> np.ndarray:
    return refund_count / (sales_count + 1e-6)

In [85]:
def create_dataframe(transactions):
    # Lager en tabell med transaksjonene
    df = pd.DataFrame(transactions)
    

    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["price"] = pd.to_numeric(df["price"])


    # Sorterer transaksjonene fra gamle til nyeste
    df = df.sort_values("timestamp", ascending=False)


    # Sett timestampe som indeksen
    df = df.set_index("timestamp").sort_index()


    
    return df

In [86]:
create_dataframe(log_parser_funksjon(pathlib.Path("data/2026/library_cafe/POS_01/register_sales.log")))

,POS,item,currency,status,price
timestamp,,,,,
2026-10-15 07:00:36,[POS_01],Cappuccino,[NOK],[SUCCESS],4.38
2026-10-15 07:01:02,[POS_01],Croissant,[EUR],[SUCCESS],3.89
2026-10-15 07:01:56,[POS_01],Latte,[USD],[REFUND],4.97
2026-10-15 07:02:38,[POS_01],Espresso,[EUR],[CANCELLED],2.67
2026-10-15 07:04:32,[POS_01],Muffin,[NOK],[REFUND],3.45
...,...,...,...,...,...
2026-10-15 08:53:43,[POS_01],Latte,[NOK],[SUCCESS],4.40
2026-10-15 08:55:52,[POS_01],Muffin,[EUR],[SUCCESS],3.05
2026-10-15 08:57:29,[POS_01],Sandwich,[NOK],[SUCCESS],7.71


In [87]:
def hourly_sales(data):
    data = data.copy()
    data["price"] = pd.to_numeric(data["price"])
    return data.groupby("item")["price"].resample("1h").sum()

In [88]:
hourly_sales(create_dataframe(log_parser_funksjon(pathlib.Path("data/2026/library_cafe/POS_01/register_sales.log"))))

item        timestamp          
Cappuccino  2026-10-15 07:00:00      60.48
            2026-10-15 08:00:00      28.49
Croissant   2026-10-15 07:00:00      15.77
            2026-10-15 08:00:00      11.21
Espresso    2026-10-15 07:00:00      21.30
            2026-10-15 08:00:00      26.74
Latte       2026-10-15 07:00:00       4.97
            2026-10-15 08:00:00      17.60
Muffin      2026-10-15 07:00:00      21.38
            2026-10-15 08:00:00      30.05
Sandwich    2026-10-15 07:00:00    2028.49
            2026-10-15 08:00:00      74.34
Name: price, dtype: float64

In [101]:
def Status_statestik(data):
    data = data.copy()

    data["status"] = data["status"].str[0]

    return data.pivot_table(
        index="item",
        columns="status",
        values="price",
        aggfunc="sum",
        fill_value=0
    ).reindex(columns=["SUCCESS", "REFUND", "CANCELLED"], fill_value=0)

In [90]:
Status_statestik(create_dataframe(log_parser_funksjon(pathlib.Path("data/2026/library_cafe/POS_01/register_sales.log"))))

status,CANCELLED,REFUND,SUCCESS
item,,,
Cappuccino,12.22,16.91,59.84
Croissant,3.92,3.99,19.07
Espresso,12.19,2.66,33.19
Latte,0.00,9.16,13.41
Muffin,7.00,7.41,37.02
Sandwich,1006.58,14.23,1082.02


In [102]:
def refund(data):
    status_pivot = Status_statestik(data)

    return status_pivot["REFUND"] / status_pivot["SUCCESS"].replace(0, np.nan)

In [92]:
refund(create_dataframe(log_parser_funksjon(pathlib.Path("data/2026/library_cafe/POS_01/register_sales.log"))))

item
Cappuccino    0.282587
Croissant     0.209229
Espresso      0.080145
Latte         0.683072
Muffin        0.200162
Sandwich      0.013151
dtype: float64

In [103]:
def full_workflow(main_path):

    main_path = pathlib.Path(main_path)

    # 1. Finn loggfiler og aktive POS-ID-er
    log_files = fin_log_fil(main_path)
    active_ids = Meta_data(main_path / "store_config.json")

    # 2. Les og parse alle loggfilene
    all_transactions = []

    for fil in log_files:
        all_transactions.extend(log_parser_funksjon(fil))

    # 3. Lag DataFrame
    df = create_dataframe(all_transactions)

    # Behold bare aktive POS-ID-er
    df = df[df["POS"].str[0].isin(active_ids)].copy()

    # 4. Fjern prisfeil med NumPy
    df = df[~np.isclose(df["price"], 999.00)].copy()

    # 5. Korriger NOK-priser
    df.loc[df["currency"].str[0] == "NOK", "price"] *= 10

    # 6. Lag rapporter
    hourly = hourly_sales(df)
    status = Status_statestik(df)
    refunds = refund(df)

    # 7. Lagre rapportene
    output_path = main_path / "output"
    output_path.mkdir(exist_ok=True)

    df.to_csv(output_path / "transactions.csv")
    hourly.to_csv(output_path / "hourly_sales.csv")
    status.to_csv(output_path / "status_statistics.csv")
    refunds.to_csv(output_path / "refund_ratio.csv")

    return df, hourly, status, refunds

In [104]:
df, hourly, status, refunds = full_workflow(pathlib.Path("data"))

print(df)
print(hourly)
print(status)
print(refunds)

                          POS        item currency       status  price
timestamp                                                             
2026-10-15 07:00:16  [POS_03]  Cappuccino    [EUR]  [CANCELLED]   4.08
2026-10-15 07:00:36  [POS_01]  Cappuccino    [NOK]    [SUCCESS]  43.80
2026-10-15 07:00:38  [POS_03]    Espresso    [EUR]    [SUCCESS]   2.56
2026-10-15 07:00:46  [POS_03]    Sandwich    [USD]  [CANCELLED]   7.98
2026-10-15 07:00:51  [POS_02]      Muffin    [USD]    [SUCCESS]   3.09
...                       ...         ...      ...          ...    ...
2026-10-15 09:05:28  [POS_02]    Espresso    [NOK]    [SUCCESS]  34.10
2026-10-15 09:06:34  [POS_01]       Latte    [NOK]    [SUCCESS]  43.70
2026-10-15 09:07:05  [POS_03]    Espresso    [EUR]     [REFUND]   3.20
2026-10-15 09:07:49  [POS_03]   Croissant    [USD]    [SUCCESS]   4.15
2026-10-15 09:09:22  [POS_03]       Latte    [EUR]    [SUCCESS]   4.12

[705 rows x 5 columns]
item        timestamp          
Cappuccino  2026-10-1